# Uplift Modelling with Meta-Learners
### Causal Data Science Portfolio — Project P3

This notebook walks through the full pipeline interactively.
All heavy lifting is in `src/` — this notebook is for exploration and presentation.

**Contents**
1. Data loading & EDA
2. Causal framework recap
3. Fit S / T / X-Learners
4. Evaluate with Qini curves
5. Targeting policy & budget optimisation
6. CATE deep-dive by segment

In [ ]:
import sys, os
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from data_loader      import load_processed, get_feature_cols
from uplift_models    import get_all_models
from evaluation       import auqc, qini_curve, random_baseline_qini
from targeting_policy import policy_summary, budget_sweep
from visualizations   import (
    plot_eda_overview, plot_uplift_distribution, plot_qini_curves,
    plot_cumulative_gain, plot_feature_importance, plot_budget_curve,
    plot_cate_heatmap, plot_model_comparison
)

print('Imports OK')

## 1. Data Loading & EDA

In [ ]:
df = load_processed()
print(f'Shape: {df.shape}')
df[['segment','visit','conversion','spend']].groupby('segment').agg(
    n=('visit','count'),
    visit_rate=('visit','mean'),
    conv_rate=('conversion','mean'),
    avg_spend=('spend','mean')
).round(4)

In [ ]:
plot_eda_overview(df)
plt.show()

## 2. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = get_feature_cols()
X         = df[feature_cols]
treatment = df['treatment_binary'].values
y         = df['conversion'].values

X_tr, X_te, t_tr, t_te, y_tr, y_te = train_test_split(
    X, treatment, y, test_size=0.30, random_state=42, stratify=treatment
)
print(f'Train: {len(X_tr):,}  |  Test: {len(X_te):,}')
print(f'Treatment rate — train: {t_tr.mean():.3f}  test: {t_te.mean():.3f}')

## 3. Fit All Meta-Learners

In [ ]:
models = get_all_models(outcome='binary')
cate_dict = {}

for model in models:
    print(f'Fitting {model.name} ...', end=' ', flush=True)
    model.fit(X_tr, t_tr, y_tr)
    cate = model.predict(X_te)
    cate_dict[model.name] = cate
    print(f'done  mean CATE={cate.mean():.5f}  std={cate.std():.5f}')

## 4. CATE Distributions

In [ ]:
plot_uplift_distribution(cate_dict)
plt.show()

## 5. Qini Curve Evaluation

In [ ]:
auqc_dict = {name: auqc(y_te, t_te, cate) for name, cate in cate_dict.items()}
for name, score in sorted(auqc_dict.items(), key=lambda x: -x[1]):
    print(f'  {name:<15}  AUQC = {score:.4f}')

In [ ]:
plot_qini_curves(cate_dict, y_te, t_te)
plt.show()

In [ ]:
plot_model_comparison(auqc_dict)
plt.show()

## 6. Targeting Policy

In [ ]:
best_name = max(auqc_dict, key=auqc_dict.get)
best_cate = cate_dict[best_name]
print(f'Best model: {best_name}')

# Try multiple budget levels
for b in [0.10, 0.20, 0.30, 0.50]:
    pol = policy_summary(best_cate, y_te, t_te, budget_frac=b)
    print(f"  Budget {b*100:.0f}%: {pol['n_targeted']:,} targeted  "
          f"profit=${pol['profit_lift_usd']:,.0f}  ROI={pol['roi_pct']:.0f}%")

In [ ]:
bdf = budget_sweep(best_cate, y_te, t_te)
plot_budget_curve(bdf)
plt.show()

opt = bdf.loc[bdf['profit_lift_usd'].idxmax()]
print(f"Optimal budget: {opt['budget_frac']*100:.1f}%  →  ${opt['profit_lift_usd']:,.2f} profit")

## 7. CATE Heatmap — Who Are the Persuadables?

In [ ]:
df_test = df.iloc[y_te.shape[0]*-1:].copy()  # approximate slice for display
# Use the first len(best_cate) rows of df as the test frame
from sklearn.model_selection import train_test_split
_, _, _, _, _, _, idx_tr, idx_te = train_test_split(
    X, treatment, y, df.index,
    test_size=0.30, random_state=42, stratify=treatment
)
df_test = df.loc[idx_te].reset_index(drop=True)

plot_cate_heatmap(df_test, best_cate, model_name=best_name)
plt.show()

## 8. Feature Importance (T-Learner)

In [ ]:
t_learner = next(m for m in models if m.name == 'T-Learner')
plot_feature_importance(t_learner, feature_cols)
plt.show()

---
## Summary

| Model | AUQC |
|---|---|
| S-Learner | See cell 5 |
| T-Learner | See cell 5 |
| X-Learner | See cell 5 |

The X-Learner is theoretically optimal with unbalanced arms; with balanced 33/33/33 split as in Hillstrom, the three models perform similarly. In a real campaign with observational data (e.g. 95% control, 5% treated), X-Learner typically dominates.

**Key takeaway:** Uplift modelling shifts targeting from "who is likely to buy?" to "who will buy *because* of our campaign?" This directly translates into higher ROI by eliminating wasted spend on Sure Things.